# M0 walkthrough: one episode through ArbiterKB
Observations → KB (beliefs + evidence) → plan → score. Synthetic RoboMME-like data, CPU only (DEVELOPMENT.md §6 M0). Each cell says what goes in and out.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.environ['ARBITER_DEBUG'] = '1'
from arbiter_kb.sources.synthetic import SyntheticSource, COVER_T
from arbiter_kb.sources.corrupt import CorruptedSource
from arbiter_kb.systems.arbiter_kb import ArbiterKBSystem
from arbiter_kb.systems.baselines import TrackerSystem
from arbiter_kb.views import text_view, ask
from arbiter_kb.debug import explain
from arbiter_kb.evaluate import run
from arbiter_kb.params import Params
params = Params.load('../params.json')  # the frozen parameter set (E5c)

In [ ]:
# Perception stand-in: a Source yields one FrameObs per frame (visible / xyz in m / uv in px per object)
src = SyntheticSource()
ep = 1025
meta, gt = src.meta(ep), src.ground_truth(ep)
frames = list(src.frames(ep))
print(meta.task_goal, '| frames:', len(frames))
print('truth:', gt.cube_to_bin, '| t*:', gt.t_star)
frames[COVER_T].objects[gt.targets[0]]

In [ ]:
# Fast loop + KB: step frame by frame; print the text view (what the arbiter will read)
s = ArbiterKBSystem(params)
s.reset(meta)
show = {COVER_T, 120, len(frames) - 1}
for f in frames:
    s.step(f)
    if f.t in show:
        print(f'--- t={f.t}')
        print(text_view(s.kb))

In [ ]:
# Evidence log: numbered so beliefs can cite it
for e in s.kb.evidence:
    if e.kind in ('disappear', 'near_pass'):
        print(e.id, e.t, e.kind, e.args, {k: v for k, v in e.data.items() if k != 'obs'})

In [ ]:
# Predicate view with the course's unify / ASK: answers carry probabilities
cube = gt.targets[0]
ask(s.kb, ('Inside', cube, '?c'))

In [ ]:
# Planner -> skill -> 4 waypoints [x, y, z, roll, pitch, yaw, gripper], positions from the KB's L0 only
step = s.plan(cube)
print(step.skill, step.target, round(step.conf, 3), step.decision)
step.waypoints.round(3)

In [ ]:
# Why does it believe that? The support chain down to raw observations (E2)
print(explain(s.trace_rows(), cube))

**Same episode with a tracker identity swap.** After the closest approach the two labels are exchanged. Bound follows the wrong label with confidence 1. ArbiterKB moved only a little mass to the other container, so it is ALSO wrong with confidence above τ_hi = 0.8: a silent failure. Since M1, R6 uses the 3D distance of the pass, so at lane 0.07 it moves almost no mass. (This corruption teleports the label ~12 cm in one frame, which no real tracker does: sanity only; see NOTES and m1_walkthrough.)

In [ ]:
bad = CorruptedSource(src, p=1.0)
rows, traces, _ = run([TrackerSystem('bound', params), ArbiterKBSystem(params)], bad, [ep], params, 'id_swap')
print(bad.log[ep])
[(r['system'], r['pred'], r['truth'], r['p_max'], r['correct']) for r in rows]

In [ ]:
# Summary figure from the full run (make selftest writes it)
from IPython.display import Image
Image('../figures/m0_belief_timeline_synthetic_p1.0.png')